# ASD Scanpath Classifier Training — all models, all configs

One notebook to train every model explored for the ASD eye-tracking paper:
- **12 PyTorch configs**: 2 backbones (MobileNetV4-small, ConvNeXtV2-tiny) x 3
  GhostNet insertion variants (PA1/PA2/PA3) x with/without CBAM attention.
- **1 TensorFlow config**: the accuracy-focused InceptionV3 track (97% reported
  accuracy in the paper).

All model/training code lives in `src/asd_train/` — this notebook only picks
configs and calls into it. See `README.md` for what each config means and
its reported accuracy.

**Framework note:** TensorFlow and PyTorch both try to claim the GPU. Run the
PyTorch section OR the TensorFlow section in a given kernel session, not both
back to back without a restart.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    # Colab's cwd is always /content regardless of where this .ipynb lives on
    # Drive -- point REPO_ROOT at the Drive-mirrored code/ folder explicitly
    # (see sync_to_drive.sh) instead of relying on Path.cwd(). This also
    # makes OUTPUT_DIR land on Drive, so results/weights survive a runtime
    # disconnect instead of vanishing with the ephemeral /content disk.
    REPO_ROOT = Path("/content/drive/MyDrive/ASD/ML4Autism/train/code")
    DATASET_ROOT = "/content/drive/MyDrive/ASD/ML4Autism/train/datasets/mahmoud-dataset"
else:
    # Local run: dataset lives in the meta-repo's data/ (sibling of this
    # train/ repo, NOT inside it) -- see ASD/.agents/record.md (T4) for how
    # it's populated (a local copy, not the slow Drive FUSE mount).
    REPO_ROOT = Path.cwd()
    DATASET_ROOT = str((REPO_ROOT / ".." / "data" / "mahmoud-dataset").resolve())

sys.path.insert(0, str(REPO_ROOT / "src"))
OUTPUT_DIR = str(REPO_ROOT / "artifacts")
print(f"REPO_ROOT    = {REPO_ROOT}")
print(f"DATASET_ROOT = {DATASET_ROOT}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")

In [ ]:
if IS_COLAB:
    # Colab preinstalls torch/torchvision/tensorflow/scikit-learn/scipy
    # already matched to its GPU driver -- don't reinstall/pin those here,
    # just add what's actually missing.
    %pip install -q timm keras-tuner

## PyTorch — Ghost / Ghost+CBAM (12 configs)

`PRESETS` has one `GhostRunConfig` per config, with every hyperparameter
read directly from the original notebooks (see `src/asd_train/configs.py`
docstring).

In [2]:
from asd_train.configs import PRESETS, BEST_PRESETS

print(f"{len(PRESETS)} configs available:")
for name in PRESETS:
    print(" -", name)

12 configs available:
 - ghost_mobilenetv4_pa1
 - ghost_mobilenetv4_pa2
 - ghost_mobilenetv4_pa3
 - ghost_convnexttiny_pa1
 - ghost_convnexttiny_pa2
 - ghost_convnexttiny_pa3
 - ghostcbam_mobilenetv4_pa1
 - ghostcbam_mobilenetv4_pa2
 - ghostcbam_mobilenetv4_pa3
 - ghostcbam_convnexttiny_pa1
 - ghostcbam_convnexttiny_pa2
 - ghostcbam_convnexttiny_pa3


### Run a single config

In [ ]:
from asd_train.train_torch import run_torch_experiment

config_name = "ghost_mobilenetv4_pa2"  # <- change to any key in PRESETS
result = run_torch_experiment(PRESETS[config_name], dataset_root=DATASET_ROOT, output_dir=OUTPUT_DIR)
result

### Run all 12 configs with grouped 5-fold CV (T6 — primary reported protocol)

Each config trains+evaluates 5 times (once per fold, subject-independent,
every participant used as test exactly once across the 5 folds). Reports
mean ± 95% CI per config, not a single hold-out number. Results are also
written incrementally to `artifacts/{config_name}_cv.json` per fold, so a
crash partway through doesn't lose already-finished folds/configs.

In [ ]:
from asd_train.train_torch import run_torch_cv_experiment

cv_results = {}
for name, cfg in PRESETS.items():
    print(f"\n{'='*80}\n{name}\n{'='*80}")
    cv_results[name] = run_torch_cv_experiment(cfg, dataset_root=DATASET_ROOT, output_dir=OUTPUT_DIR)

for name, r in cv_results.items():
    print(
        f"{name:35s} "
        f"acc={r.mean_accuracy:.4f} (95% CI {r.ci_accuracy[0]:.4f}-{r.ci_accuracy[1]:.4f})  "
        f"sens={r.mean_sensitivity:.4f}  spec={r.mean_specificity:.4f}  "
        f"auc={r.mean_auc_roc:.4f} (95% CI {r.ci_auc_roc[0]:.4f}-{r.ci_auc_roc[1]:.4f})"
    )

## TensorFlow — InceptionV3 (97% reported accuracy)

**Restart the kernel before running this section** if you already ran the
PyTorch cells above in this session.

In [ ]:
import sys
from pathlib import Path

from asd_train.configs import INCEPTIONV3_CONFIG
from asd_train.train_tf import run_inceptionv3_experiment

# Re-set after kernel restart -- same Colab-detection logic as the Setup cell.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/drive/MyDrive/ASD/ML4Autism/train/code")
    DATASET_ROOT = "/content/drive/MyDrive/ASD/ML4Autism/train/datasets/mahmoud-dataset"
except ImportError:
    REPO_ROOT = Path.cwd()
    DATASET_ROOT = str((REPO_ROOT / ".." / "data" / "mahmoud-dataset").resolve())

sys.path.insert(0, str(REPO_ROOT / "src"))
OUTPUT_DIR = str(REPO_ROOT / "artifacts")
print(f"DATASET_ROOT = {DATASET_ROOT}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")

tf_result = run_inceptionv3_experiment(INCEPTIONV3_CONFIG, dataset_root=DATASET_ROOT, output_dir=OUTPUT_DIR)
tf_result